# Propensity Score Methods: Matching, Weighting, and Doubly Robust Estimation

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


**Business setting:** treatment is **self-selected at the unit level** and you have rich pre-treatment covariates. Examples: customers who joined the loyalty program vs. not; merchants who adopted the new POS feature vs. not; stores whose managers opted into the training.

**Identifying assumptions:**

1. **Conditional ignorability (no unmeasured confounders):** given covariates $X$, treatment is as good as random: $\{Y(0), Y(1)\} \perp D \mid X$. This is a *strong, untestable* assumption — everything that drives both adoption and the outcome must be measured.
2. **Overlap (positivity):** $0 < P(D=1 \mid X) < 1$. Every type of unit must have some chance of being in both groups.

## Simulated example: loyalty-program adoption

In [ ]:
n = 8000
tenure   = rng.gamma(3, 1.2, n)                  # years as customer
spend0   = rng.lognormal(4.0, 0.5, n)            # baseline monthly spend
engage   = rng.beta(2, 5, n) * 10                # app engagement score
urban    = rng.binomial(1, 0.55, n)

# adoption depends on observables (heavier spenders & engaged users join)
logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
p = 1 / (1 + np.exp(-logit))
D = rng.binomial(1, p)

TRUE_ATT = 12.0     # loyalty program adds $12/month for adopters
y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
     + TRUE_ATT * D + rng.normal(0, 12, n))

cust = pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                         urban=urban, D=D, y=y))
naive = cust.loc[cust.D==1, "y"].mean() - cust.loc[cust.D==0, "y"].mean()
print(f"Naive difference: {naive:.1f}   (true ATT = {TRUE_ATT})")

The naive gap is far above 12 because adopters were already higher-spending.

## Estimate the propensity score and check overlap

In [ ]:
from sklearn.linear_model import LogisticRegression

X_cols = ["tenure", "spend0", "engage", "urban"]
ps_model = LogisticRegression(max_iter=1000).fit(cust[X_cols], cust.D)
cust["ps"] = ps_model.predict_proba(cust[X_cols])[:, 1]

plt.figure(figsize=(8, 3.5))
sns.kdeplot(data=cust, x="ps", hue="D", common_norm=False, fill=True)
plt.title("Propensity score overlap"); plt.tight_layout(); plt.show()

If the distributions barely overlap, no propensity method can save you — the data simply don't contain the comparison you need. Standard fixes: trim to the common-support region (e.g., ps ∈ [0.05, 0.95]) and re-define the estimand as the effect *within* that population, saying so explicitly.

## Nearest-neighbor matching on the propensity score

In [ ]:
from sklearn.neighbors import NearestNeighbors

treated  = cust[cust.D == 1]
control  = cust[cust.D == 0]
nn = NearestNeighbors(n_neighbors=1).fit(control[["ps"]])
dist, idx = nn.kneighbors(treated[["ps"]])

# caliper: drop matches farther than 0.2 SD of the logit of ps
logit_ps = np.log(cust.ps / (1 - cust.ps))
caliper = 0.2 * logit_ps.std()
keep = (dist.ravel() < caliper)
att_match = (treated.y.values[keep]
             - control.y.values[idx.ravel()][keep]).mean()
print(f"PS matching ATT: {att_match:.2f}  ({keep.mean():.0%} matched)")

## Balance diagnostics — the test that matters

Whether matching/weighting "worked" is judged by **covariate balance**, conventionally absolute standardized mean differences (SMD) < 0.1 after adjustment:

In [ ]:
def smd(df, w=None):
    out = {}
    for c in X_cols:
        if w is None:
            m1, m0 = df.loc[df.D==1, c].mean(), df.loc[df.D==0, c].mean()
            v1, v0 = df.loc[df.D==1, c].var(),  df.loc[df.D==0, c].var()
        else:
            w1, w0 = w[df.D==1], w[df.D==0]
            m1 = np.average(df.loc[df.D==1, c], weights=w1)
            m0 = np.average(df.loc[df.D==0, c], weights=w0)
            v1 = np.average((df.loc[df.D==1, c]-m1)**2, weights=w1)
            v0 = np.average((df.loc[df.D==0, c]-m0)**2, weights=w0)
        out[c] = (m1 - m0) / np.sqrt((v1 + v0) / 2)
    return pd.Series(out)

print("Before adjustment:\n", smd(cust).round(3))

## Inverse probability weighting (IPW) for the ATT

For the ATT, weight treated units by 1 and controls by $\frac{ps}{1-ps}$ — reweighting controls to look like the treated population:

In [ ]:
w = np.where(cust.D == 1, 1.0, cust.ps / (1 - cust.ps))
att_ipw = (np.average(cust.loc[cust.D==1, "y"])
           - np.average(cust.loc[cust.D==0, "y"],
                        weights=w[cust.D==0]))
print(f"IPW ATT: {att_ipw:.2f}")
print("After weighting:\n", smd(cust, w=w).round(3))

Inspect the weight distribution — a handful of huge weights means near-violations of overlap and unstable estimates. Stabilized or truncated weights (cap at the 99th percentile) are common pragmatic fixes; report both.

## Doubly robust estimation (AIPW)

Doubly robust estimators combine an outcome regression with IPW and are consistent if **either** model is right. This is the recommended default; `DoubleML` and `econml` implement cross-fitted versions (Section 12 shows the full ML pipeline):

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from doubleml import DoubleMLData, DoubleMLIRM
from sklearn.ensemble import GradientBoostingClassifier

dml_data = DoubleMLData(cust, y_col="y", d_cols="D", x_cols=X_cols)
irm = DoubleMLIRM(dml_data,
                  ml_g=GradientBoostingRegressor(),
                  ml_m=GradientBoostingClassifier(),
                  score="ATTE",            # ATT version of the score
                  n_folds=5)
irm.fit()
print(irm.summary)

## Honest limits

Propensity methods adjust **only for what you measured**. They do nothing about unobserved confounders — the manager's ambition, the customer's life event. In business practice the credibility of a propensity analysis rests on (a) a rich pre-treatment feature set, ideally including the *lagged outcome itself*, (b) balance diagnostics, and (c) a formal sensitivity analysis for unmeasured confounding (Section 13). If selection is plausibly on *unobservables*, look for an instrument (Section 11) or a discontinuity (Section 10) instead.